# Region Captioning Demo

This notebook segments a region with SAM3, crops it, and captions the crop with Florence-2 and BLIP.


In [ ]:
%matplotlib inline

import os
import sys
import tempfile
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATASET_DIR = ROOT / 'datasets' / 'coco' / 'val2017'
OUTPUT_DIR = ROOT / 'outputs'
ASSET_DIR = ROOT / 'assets'
HF_CACHE = Path.home() / '.cache' / 'huggingface' / 'hub'
HF_MODULES_CACHE = Path(tempfile.gettempdir()) / 'hf_modules'
os.environ['HF_MODULES_CACHE'] = str(HF_MODULES_CACHE)
HF_MODULES_CACHE.mkdir(parents=True, exist_ok=True)

for path in (DATASET_DIR, OUTPUT_DIR, ASSET_DIR):
    path.mkdir(parents=True, exist_ok=True)

print('Project root:', ROOT)
print('COCO folder:', DATASET_DIR)
print('Output folder:', OUTPUT_DIR)
print('HF cache:', HF_CACHE)
print('HF modules cache:', HF_MODULES_CACHE)


In [ ]:
# Optional: switch to True on a fresh machine.
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    import subprocess
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'pillow',
        'matplotlib',
        'requests',
        'accelerate',
        'sentencepiece',
        'ipywidgets',
        'timm',
        'einops',
        'transformers==4.49.0',
        'huggingface_hub<1.0',
        'tokenizers<0.22',
        'git+https://github.com/facebookresearch/sam3.git',
    ])
    print('Dependencies installed.')
else:
    print('Skipping dependency installation. Set INSTALL_DEPENDENCIES = True if needed.')


In [ ]:
import urllib.request
from urllib.error import URLError

SAMPLE_COCO_URL = 'http://images.cocodataset.org/val2017/000000039769.jpg'
SAMPLE_IMAGE_PATH = DATASET_DIR / '000000039769.jpg'
FALLBACK_IMAGE_PATH = ASSET_DIR / 'example_input.jpg'

if not SAMPLE_IMAGE_PATH.exists():
    try:
        urllib.request.urlretrieve(SAMPLE_COCO_URL, SAMPLE_IMAGE_PATH)
        print('Downloaded sample COCO image to', SAMPLE_IMAGE_PATH)
    except URLError as exc:
        print('COCO download skipped:', exc)

if SAMPLE_IMAGE_PATH.exists():
    IMAGE_PATH = SAMPLE_IMAGE_PATH
    TEXT_PROMPT = 'cat'
    IMAGE_SOURCE = 'COCO sample'
else:
    IMAGE_PATH = FALLBACK_IMAGE_PATH
    TEXT_PROMPT = 'bottle'
    IMAGE_SOURCE = 'local fallback asset'

print('Using image:', IMAGE_PATH)
print('Image source:', IMAGE_SOURCE)
print('Text prompt:', TEXT_PROMPT)


In [ ]:
import sys
import types
import importlib.machinery

import torch
from PIL import Image, ImageDraw
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download, snapshot_download
from transformers import AutoModelForCausalLM, AutoProcessor, BlipForConditionalGeneration, BlipProcessor


def install_sam3_compat_shims():
    if 'triton' not in sys.modules:
        triton_module = types.ModuleType('triton')
        triton_module.__spec__ = importlib.machinery.ModuleSpec('triton', loader=None)

        def jit(fn=None, **_kwargs):
            if fn is None:
                def decorator(inner):
                    return inner
                return decorator
            return fn

        def autotune(configs=None, key=None):
            def decorator(fn):
                return fn
            return decorator

        def cdiv(x, y):
            return (x + y - 1) // y

        class Config:
            def __init__(self, *args, **kwargs):
                self.args = args
                self.kwargs = kwargs

        triton_module.jit = jit
        triton_module.autotune = autotune
        triton_module.cdiv = cdiv
        triton_module.Config = Config
        sys.modules['triton'] = triton_module

    if 'triton.language' not in sys.modules:
        triton_language = types.ModuleType('triton.language')
        triton_language.__spec__ = importlib.machinery.ModuleSpec('triton.language', loader=None)

        class constexpr:
            pass

        triton_language.constexpr = constexpr
        sys.modules['triton.language'] = triton_language
        sys.modules['triton'].language = triton_language

    if 'decord' not in sys.modules:
        decord_module = types.ModuleType('decord')
        decord_module.__spec__ = importlib.machinery.ModuleSpec('decord', loader=None)

        def cpu(index=0):
            return index

        class VideoReader:
            def __init__(self, *args, **kwargs):
                raise RuntimeError('decord is not installed in this environment. VideoReader is unavailable.')

        decord_module.cpu = cpu
        decord_module.VideoReader = VideoReader
        sys.modules['decord'] = decord_module

    if 'pycocotools' not in sys.modules:
        pycocotools_module = types.ModuleType('pycocotools')
        pycocotools_module.__spec__ = importlib.machinery.ModuleSpec('pycocotools', loader=None)
        sys.modules['pycocotools'] = pycocotools_module

    if 'pycocotools.mask' not in sys.modules:
        pycocotools_mask = types.ModuleType('pycocotools.mask')
        pycocotools_mask.__spec__ = importlib.machinery.ModuleSpec('pycocotools.mask', loader=None)

        def _missing_attr(name):
            raise RuntimeError(f'pycocotools.mask.{name} is unavailable in this environment.')

        pycocotools_mask.__getattr__ = _missing_attr
        sys.modules['pycocotools.mask'] = pycocotools_mask
        sys.modules['pycocotools'].mask = pycocotools_mask


install_sam3_compat_shims()

from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)


def cached_snapshot(repo_dir_name: str, required_files=None):
    required_files = required_files or []
    snapshots_dir = HF_CACHE / repo_dir_name / 'snapshots'
    if snapshots_dir.exists():
        snapshots = sorted([p for p in snapshots_dir.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
        for snapshot in snapshots:
            if all((snapshot / name).exists() for name in required_files):
                return snapshot
    return None


def resolve_snapshot(repo_id: str, repo_dir_name: str, required_files=None):
    cached = cached_snapshot(repo_dir_name, required_files=required_files)
    if cached is not None:
        print(f'Using cached snapshot for {repo_id}:', cached)
        return cached
    print(f'Downloading snapshot for {repo_id} ...')
    return Path(snapshot_download(repo_id=repo_id, local_files_only=False))


def resolve_file(repo_id: str, repo_dir_name: str, filename: str):
    cached = cached_snapshot(repo_dir_name, required_files=[filename])
    if cached is not None:
        candidate = cached / filename
        if candidate.exists():
            print(f'Using cached file for {repo_id}:', candidate)
            return candidate
    print(f'Downloading {filename} from {repo_id} ...')
    return Path(hf_hub_download(repo_id=repo_id, filename=filename, local_files_only=False))


In [ ]:
sam_checkpoint = resolve_file('facebook/sam3', 'models--facebook--sam3', 'sam3.pt')
florence_snapshot = resolve_snapshot(
    'microsoft/Florence-2-large-ft',
    'models--microsoft--Florence-2-large-ft',
    required_files=['config.json', 'generation_config.json', 'preprocessor_config.json'],
)
blip_snapshot = resolve_snapshot(
    'Salesforce/blip-image-captioning-base',
    'models--Salesforce--blip-image-captioning-base',
    required_files=['config.json', 'preprocessor_config.json'],
)

sam_model = build_sam3_image_model(
    checkpoint_path=str(sam_checkpoint),
    load_from_HF=False,
    device=device,
)
sam_processor = Sam3Processor(sam_model)

florence_processor = AutoProcessor.from_pretrained(florence_snapshot, trust_remote_code=True)
florence_model = AutoModelForCausalLM.from_pretrained(
    florence_snapshot,
    trust_remote_code=True,
).eval().to(device)

blip_processor = BlipProcessor.from_pretrained(blip_snapshot)
blip_model = BlipForConditionalGeneration.from_pretrained(blip_snapshot).eval().to(device)

print('SAM3, Florence-2, and BLIP are ready.')


In [ ]:
image = Image.open(IMAGE_PATH).convert('RGB')
print('Image size:', image.size)
image


In [ ]:
state = sam_processor.set_image(image)
sam_output = sam_processor.set_text_prompt(state=state, prompt=TEXT_PROMPT)

boxes = sam_output['boxes']
scores = sam_output['scores']
num_boxes = int(boxes.shape[0]) if hasattr(boxes, 'shape') else len(boxes)
print('Detected regions:', num_boxes)

if num_boxes == 0:
    raise ValueError(
        f'No region found for prompt: {TEXT_PROMPT}. '
        'Try another prompt that matches the image contents.'
    )

best_index = int(scores.argmax()) if hasattr(scores, 'argmax') else 0
bbox = boxes[best_index].tolist() if hasattr(boxes[best_index], 'tolist') else boxes[best_index]
width, height = image.size
x1, y1, x2, y2 = bbox
x1 = max(0, min(int(x1), width - 1))
y1 = max(0, min(int(y1), height - 1))
x2 = max(x1 + 1, min(int(x2), width))
y2 = max(y1 + 1, min(int(y2), height))

overlay = image.copy()
draw = ImageDraw.Draw(overlay)
draw.rectangle([x1, y1, x2, y2], outline='red', width=4)

crop = image.crop((x1, y1, x2, y2))
crop_path = OUTPUT_DIR / 'sam_crop.jpg'
overlay_path = OUTPUT_DIR / 'sam_overlay.jpg'
image_path_copy = OUTPUT_DIR / 'selected_input.jpg'

crop.save(crop_path)
overlay.save(overlay_path)
image.save(image_path_copy)

print('Saved selected input to', image_path_copy)
print('Saved crop to', crop_path)
print('Saved overlay to', overlay_path)
print('Crop size:', crop.size)
overlay


In [ ]:
florence_prompt = '<MORE_DETAILED_CAPTION>'
florence_inputs = florence_processor(text=florence_prompt, images=crop, return_tensors='pt')
florence_inputs = {k: v.to(device) for k, v in florence_inputs.items()}

with torch.no_grad():
    florence_ids = florence_model.generate(**florence_inputs, max_new_tokens=80)

florence_caption = florence_processor.batch_decode(florence_ids, skip_special_tokens=True)[0]
print('Florence-2 caption:', florence_caption)


In [ ]:
blip_inputs = blip_processor(images=crop, return_tensors='pt')
blip_inputs = {k: v.to(device) for k, v in blip_inputs.items()}

with torch.no_grad():
    blip_ids = blip_model.generate(**blip_inputs, max_new_tokens=50)

blip_caption = blip_processor.decode(blip_ids[0], skip_special_tokens=True)
print('BLIP caption:', blip_caption)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(image)
axes[0].set_title(f'Original image ({IMAGE_SOURCE})')
axes[0].axis('off')

axes[1].imshow(overlay)
axes[1].set_title(f'SAM3 region for: {TEXT_PROMPT}')
axes[1].axis('off')

axes[2].imshow(crop)
axes[2].set_title('Cropped region')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print('Florence-2:', florence_caption)
print('BLIP:', blip_caption)
